# GASP Noise Robustness Analysis

This notebook evaluates the robustness of GASP methods to noise across various conditions.

## Analyses
1. **Noise vs Methods**: Compare linear, affine, quad, quad-cross across noise levels
2. **Phase Cycle Count**: Effect of 2, 4, 8, 16 phase cycles on noise robustness
3. **Cross-Tissue Generalization**: How noise affects performance when training on one T1/T2 and testing on others

## Tissue Types
- Gray Matter, White Matter, CSF/Water, Muscle, Fat, Liver

In [ ]:
import sys
sys.path.insert(0, '../../')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# GASP imports
from gasp import responses, sampling
from gasp.gasp import train_gasp, run_gasp
from gasp.simulation import SSFPParams, simulate_ssfp_simple

# Set random seed for reproducibility
np.random.seed(42)

# Set default plot style
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Imports successful!")

## 1. Configuration Parameters

In [ ]:
# Simulation parameters
WIDTH = 256  # Number of off-resonance points
HEIGHT = 1   # 1D simulation
TRS_BASE = [5e-3, 10e-3, 20e-3]  # 3 TR values
GRADIENT = 2 * np.pi
ALPHA = np.deg2rad(30)  # Flip angle

# Phase cycle counts to test
PC_COUNTS = [2, 4, 8, 16]
DEFAULT_NPCS = 8  # Default for other analyses

# Target response parameters (bandpass)
BANDWIDTH = 0.25
SHIFT = 0.0

# Noise levels to test
NOISE_SIGMAS = np.array([0.0, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1])

# Monte Carlo iterations
N_MONTE_CARLO = 25

# GASP methods
GASP_METHODS = ['linear', 'affine', 'quad', 'quad-cross']

# Tissue types with T1/T2 parameters (seconds)
TISSUES = {
    'Water/CSF': {'T1': 4.0, 'T2': 2.0, 'ratio': 2.0},
    'Fat': {'T1': 0.25, 'T2': 0.07, 'ratio': 3.57},
    'White Matter': {'T1': 0.6, 'T2': 0.08, 'ratio': 7.5},
    'Gray Matter': {'T1': 0.9, 'T2': 0.1, 'ratio': 9.0},
    'Liver': {'T1': 0.5, 'T2': 0.04, 'ratio': 12.5},
    'Muscle': {'T1': 0.9, 'T2': 0.05, 'ratio': 18.0},
}

print(f"Testing {len(NOISE_SIGMAS)} noise levels: {NOISE_SIGMAS}")
print(f"Testing {len(PC_COUNTS)} PC counts: {PC_COUNTS}")
print(f"Testing {len(TISSUES)} tissue types")
print(f"Monte Carlo iterations: {N_MONTE_CARLO}")

## 2. Setup Target Response (Bandpass)

In [ ]:
# Create bandpass target response
D = responses.bandpass(WIDTH, bw=BANDWIDTH, shift=SHIFT, type='butterworth')

print(f"Target response shape: {D.shape}")

# Plot target response
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Compare response types
D_square = responses.square(WIDTH, bw=BANDWIDTH, shift=SHIFT)
D_gaussian = responses.gaussian(WIDTH, bw=BANDWIDTH, shift=SHIFT)
D_bandpass = responses.bandpass(WIDTH, bw=BANDWIDTH, shift=SHIFT, type='butterworth')

axes[0].plot(D_square, 'b-', linewidth=2, label='Square')
axes[0].set_title('Square Response')
axes[0].set_xlabel('Off-resonance index')
axes[0].set_ylabel('Magnitude')

axes[1].plot(D_gaussian, 'g-', linewidth=2, label='Gaussian')
axes[1].set_title('Gaussian Response')
axes[1].set_xlabel('Off-resonance index')

axes[2].plot(D_bandpass, 'r-', linewidth=2, label='Butterworth Bandpass')
axes[2].set_title('Butterworth Bandpass (Used)')
axes[2].set_xlabel('Off-resonance index')

plt.suptitle(f'Target Response Comparison (bandwidth={BANDWIDTH})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Use bandpass for analysis
D = D_bandpass

## 3. Helper Functions

In [ ]:
def create_params(npcs):
    """Create acquisition parameters for given number of phase cycles."""
    n_points = npcs * len(TRS_BASE)
    TRs, PCs = sampling.grid_TR_sampling(n_points=n_points, TRs=TRS_BASE)
    return SSFPParams(n_points, ALPHA, TRs, PCs)


def generate_signal(T1, T2, params):
    """Generate bSSFP signal for given tissue and acquisition parameters."""
    M = simulate_ssfp_simple(
        width=WIDTH, height=HEIGHT,
        T1=T1, T2=T2,
        params=params,
        minTR=np.min(TRS_BASE),
        gradient=GRADIENT
    )
    return M


def add_complex_noise(signal, sigma):
    """Add complex Gaussian noise to signal."""
    if sigma == 0:
        return signal.copy()
    noise = sigma * (np.random.randn(*signal.shape) + 1j * np.random.randn(*signal.shape))
    return signal + noise


def compute_metrics(output, target):
    """Compute performance metrics."""
    output_mag = np.abs(output).flatten()
    target_flat = target.flatten()
    
    rmse = np.sqrt(np.mean((output_mag - target_flat) ** 2))
    
    if np.std(output_mag) > 1e-10 and np.std(target_flat) > 1e-10:
        corr = np.corrcoef(output_mag, target_flat)[0, 1]
    else:
        corr = 0.0
    
    return {'rmse': rmse, 'correlation': corr}


print("Helper functions defined.")

## 4. Sanity Check - Verify Signal Generation

In [ ]:
# Test with Gray Matter
test_params = create_params(DEFAULT_NPCS)
test_tissue = TISSUES['Gray Matter']

M_test = generate_signal(T1=test_tissue['T1'], T2=test_tissue['T2'], params=test_params)
print(f"Signal shape: {M_test.shape}")

# Train GASP
I_test, A_test = train_gasp(M_test, D, method='affine', useL2=True, lam=1e-2)
print(f"Output shape: {I_test.shape}")

metrics = compute_metrics(I_test, D)
print(f"Clean signal - RMSE: {metrics['rmse']:.4f}, Correlation: {metrics['correlation']:.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(np.abs(M_test[0, :, 0]), label='PC 1')
axes[0].plot(np.abs(M_test[0, :, -1]), label=f'PC {M_test.shape[-1]}')
axes[0].set_xlabel('Off-resonance index')
axes[0].set_ylabel('Signal magnitude')
axes[0].set_title('bSSFP Signal (Gray Matter)')
axes[0].legend()

axes[1].plot(D, 'k--', label='Target', linewidth=2)
axes[1].plot(np.abs(I_test).flatten(), 'b-', label='GASP Output', linewidth=2)
axes[1].set_xlabel('Off-resonance index')
axes[1].set_ylabel('Magnitude')
axes[1].set_title(f'GASP Output (RMSE={metrics["rmse"]:.4f})')
axes[1].legend()

plt.tight_layout()
plt.show()

---
# Part A: Effect of Phase Cycle Count on Noise Robustness

Test how the number of phase cycles (2, 4, 8, 16) affects noise robustness.

In [ ]:
def analyze_pc_count_effect(tissue_name, tissue_params, pc_counts=PC_COUNTS,
                            noise_sigmas=NOISE_SIGMAS, n_mc=N_MONTE_CARLO,
                            method='affine'):
    """
    Analyze effect of phase cycle count on noise robustness.
    """
    results = {pc: {'rmse': [], 'correlation': []} for pc in pc_counts}
    
    for npcs in pc_counts:
        params = create_params(npcs)
        M_clean = generate_signal(T1=tissue_params['T1'], T2=tissue_params['T2'], params=params)
        
        for sigma in noise_sigmas:
            rmse_list, corr_list = [], []
            
            for _ in range(n_mc):
                M_noisy = add_complex_noise(M_clean, sigma)
                
                try:
                    I, _ = train_gasp(M_noisy, D, method=method, useL2=True, lam=1e-2)
                    metrics = compute_metrics(I, D)
                    rmse_list.append(metrics['rmse'])
                    corr_list.append(metrics['correlation'])
                except:
                    rmse_list.append(np.nan)
                    corr_list.append(np.nan)
            
            results[npcs]['rmse'].append(np.nanmean(rmse_list))
            results[npcs]['correlation'].append(np.nanmean(corr_list))
    
    return results

print("PC count analysis function defined.")

In [ ]:
# Run PC count analysis for multiple tissues
test_tissues = ['Gray Matter', 'Muscle', 'Water/CSF']
pc_results = {}

print("Analyzing effect of phase cycle count...")
for tissue_name in tqdm(test_tissues, desc="Tissues"):
    pc_results[tissue_name] = analyze_pc_count_effect(
        tissue_name, TISSUES[tissue_name]
    )

print("Done!")

In [ ]:
# Plot PC count effect
fig, axes = plt.subplots(len(test_tissues), 2, figsize=(14, 4*len(test_tissues)))

pc_colors = cm.viridis(np.linspace(0.2, 0.9, len(PC_COUNTS)))

for idx, tissue_name in enumerate(test_tissues):
    results = pc_results[tissue_name]
    
    for i, npcs in enumerate(PC_COUNTS):
        n_acq = npcs * len(TRS_BASE)
        label = f'{npcs} PCs ({n_acq} acq)'
        
        axes[idx, 0].semilogy(NOISE_SIGMAS, results[npcs]['rmse'], 'o-',
                              color=pc_colors[i], label=label, linewidth=2)
        axes[idx, 1].plot(NOISE_SIGMAS, results[npcs]['correlation'], 'o-',
                          color=pc_colors[i], label=label, linewidth=2)
    
    axes[idx, 0].set_xlabel('Noise Sigma')
    axes[idx, 0].set_ylabel('RMSE (log)')
    axes[idx, 0].set_title(f'{tissue_name} - RMSE vs Noise')
    axes[idx, 0].legend(fontsize=9)
    
    axes[idx, 1].set_xlabel('Noise Sigma')
    axes[idx, 1].set_ylabel('Correlation')
    axes[idx, 1].set_title(f'{tissue_name} - Correlation vs Noise')
    axes[idx, 1].legend(fontsize=9)
    axes[idx, 1].set_ylim([0, 1.05])

plt.suptitle('Effect of Phase Cycle Count on Noise Robustness\n(3 TRs, Affine Method, Bandpass Target)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary: PC count effect at high noise
print("\n" + "="*70)
print("PHASE CYCLE COUNT EFFECT SUMMARY (at noise sigma = 0.05)")
print("="*70)

noise_idx = 5  # sigma = 0.05

for tissue_name in test_tissues:
    print(f"\n{tissue_name}:")
    results = pc_results[tissue_name]
    
    for npcs in PC_COUNTS:
        n_acq = npcs * len(TRS_BASE)
        rmse = results[npcs]['rmse'][noise_idx]
        corr = results[npcs]['correlation'][noise_idx]
        print(f"  {npcs:2d} PCs ({n_acq:2d} acquisitions): RMSE={rmse:.4f}, Corr={corr:.4f}")

---
# Part B: Cross-Tissue Generalization Under Noise

Analyze how training on one T1/T2 ratio affects performance on other tissues, and how noise amplifies this effect.

In [ ]:
def cross_tissue_generalization_analysis(train_tissue, test_tissues, 
                                         noise_sigmas=NOISE_SIGMAS,
                                         n_mc=N_MONTE_CARLO, method='affine'):
    """
    Train on one tissue, test on all tissues at different noise levels.
    Returns performance relative to same-tissue performance.
    """
    params = create_params(DEFAULT_NPCS)
    
    # Generate clean training signal and train coefficients
    train_params = TISSUES[train_tissue]
    M_train = generate_signal(T1=train_params['T1'], T2=train_params['T2'], params=params)
    _, train_coeffs = train_gasp(M_train, D, method=method, useL2=True, lam=1e-2)
    
    results = {tissue: {'rmse': [], 'correlation': [], 'ratio_diff': []} 
               for tissue in test_tissues}
    
    train_ratio = train_params['ratio']
    
    for test_tissue in test_tissues:
        test_params = TISSUES[test_tissue]
        M_test = generate_signal(T1=test_params['T1'], T2=test_params['T2'], params=params)
        
        # Store ratio difference
        ratio_diff = abs(test_params['ratio'] - train_ratio)
        
        for sigma in noise_sigmas:
            rmse_list, corr_list = [], []
            
            for _ in range(n_mc):
                M_noisy = add_complex_noise(M_test, sigma)
                
                try:
                    # Apply pre-trained coefficients
                    I = run_gasp(M_noisy, train_coeffs, method=method)
                    metrics = compute_metrics(I, D)
                    rmse_list.append(metrics['rmse'])
                    corr_list.append(metrics['correlation'])
                except:
                    rmse_list.append(np.nan)
                    corr_list.append(np.nan)
            
            results[test_tissue]['rmse'].append(np.nanmean(rmse_list))
            results[test_tissue]['correlation'].append(np.nanmean(corr_list))
        
        results[test_tissue]['ratio_diff'] = ratio_diff
        results[test_tissue]['test_ratio'] = test_params['ratio']
    
    return results, train_ratio

print("Cross-tissue analysis function defined.")

In [ ]:
# Run cross-tissue analysis for different training tissues
training_tissues = ['Gray Matter', 'Muscle', 'Water/CSF']
test_tissue_list = list(TISSUES.keys())

cross_tissue_results = {}

print("Running cross-tissue generalization analysis...")
for train_tissue in tqdm(training_tissues, desc="Training tissues"):
    results, train_ratio = cross_tissue_generalization_analysis(
        train_tissue, test_tissue_list
    )
    cross_tissue_results[train_tissue] = {
        'results': results,
        'train_ratio': train_ratio
    }

print("Done!")

In [ ]:
# Plot cross-tissue generalization
fig, axes = plt.subplots(len(training_tissues), 2, figsize=(14, 4*len(training_tissues)))

tissue_colors = cm.Set2(np.linspace(0, 1, len(test_tissue_list)))

for idx, train_tissue in enumerate(training_tissues):
    data = cross_tissue_results[train_tissue]
    results = data['results']
    train_ratio = data['train_ratio']
    
    for i, test_tissue in enumerate(test_tissue_list):
        is_same = (test_tissue == train_tissue)
        style = '-' if is_same else '--'
        lw = 3 if is_same else 1.5
        marker = 'o' if is_same else 's'
        
        label = f"{test_tissue} (r={results[test_tissue]['test_ratio']:.1f})"
        if is_same:
            label += " [TRAIN]"
        
        axes[idx, 0].semilogy(NOISE_SIGMAS, results[test_tissue]['rmse'], 
                              f'{marker}{style}', color=tissue_colors[i], 
                              label=label, linewidth=lw, markersize=5)
        axes[idx, 1].plot(NOISE_SIGMAS, results[test_tissue]['correlation'], 
                          f'{marker}{style}', color=tissue_colors[i], 
                          label=label, linewidth=lw, markersize=5)
    
    axes[idx, 0].set_xlabel('Noise Sigma')
    axes[idx, 0].set_ylabel('RMSE (log)')
    axes[idx, 0].set_title(f'Trained on: {train_tissue} (T1/T2 ratio = {train_ratio:.1f})')
    axes[idx, 0].legend(fontsize=8, loc='upper left')
    
    axes[idx, 1].set_xlabel('Noise Sigma')
    axes[idx, 1].set_ylabel('Correlation')
    axes[idx, 1].set_title(f'Trained on: {train_tissue}')
    axes[idx, 1].legend(fontsize=8, loc='lower left')
    axes[idx, 1].set_ylim([0, 1.05])

plt.suptitle('Cross-Tissue Generalization Under Noise\n(Solid = training tissue, Dashed = other tissues, r = T1/T2 ratio)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze: Performance degradation vs T1/T2 ratio difference
def compute_degradation_vs_ratio_diff(cross_tissue_results, noise_idx=5):
    """
    Compute how performance degrades as a function of T1/T2 ratio difference from training.
    """
    all_data = []
    
    for train_tissue, data in cross_tissue_results.items():
        train_ratio = data['train_ratio']
        results = data['results']
        
        # Get baseline (same-tissue) performance
        baseline_rmse_clean = results[train_tissue]['rmse'][0]
        baseline_rmse_noisy = results[train_tissue]['rmse'][noise_idx]
        
        for test_tissue, test_data in results.items():
            ratio_diff = abs(test_data['test_ratio'] - train_ratio)
            
            rmse_clean = test_data['rmse'][0]
            rmse_noisy = test_data['rmse'][noise_idx]
            corr_noisy = test_data['correlation'][noise_idx]
            
            # Relative degradation
            rel_degradation = rmse_noisy / baseline_rmse_noisy if baseline_rmse_noisy > 0 else np.inf
            
            all_data.append({
                'train_tissue': train_tissue,
                'test_tissue': test_tissue,
                'train_ratio': train_ratio,
                'test_ratio': test_data['test_ratio'],
                'ratio_diff': ratio_diff,
                'rmse_clean': rmse_clean,
                'rmse_noisy': rmse_noisy,
                'corr_noisy': corr_noisy,
                'rel_degradation': rel_degradation,
                'is_same': train_tissue == test_tissue
            })
    
    return pd.DataFrame(all_data)


degradation_df = compute_degradation_vs_ratio_diff(cross_tissue_results)

# Plot degradation vs ratio difference
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

train_colors = {'Gray Matter': 'blue', 'Muscle': 'red', 'Water/CSF': 'green'}

for train_tissue in training_tissues:
    subset = degradation_df[degradation_df['train_tissue'] == train_tissue]
    color = train_colors[train_tissue]
    
    axes[0].scatter(subset['ratio_diff'], subset['rmse_noisy'], 
                   c=color, label=f'Train: {train_tissue}', s=80, alpha=0.7)
    axes[1].scatter(subset['ratio_diff'], subset['corr_noisy'], 
                   c=color, label=f'Train: {train_tissue}', s=80, alpha=0.7)
    axes[2].scatter(subset['ratio_diff'], subset['rel_degradation'], 
                   c=color, label=f'Train: {train_tissue}', s=80, alpha=0.7)

axes[0].set_xlabel('|T1/T2 ratio difference|', fontsize=12)
axes[0].set_ylabel('RMSE at σ=0.05', fontsize=12)
axes[0].set_title('Absolute RMSE vs Ratio Difference')
axes[0].legend()

axes[1].set_xlabel('|T1/T2 ratio difference|', fontsize=12)
axes[1].set_ylabel('Correlation at σ=0.05', fontsize=12)
axes[1].set_title('Correlation vs Ratio Difference')
axes[1].legend()
axes[1].set_ylim([0, 1.05])

axes[2].set_xlabel('|T1/T2 ratio difference|', fontsize=12)
axes[2].set_ylabel('Relative Degradation', fontsize=12)
axes[2].set_title('Relative RMSE Degradation vs Ratio Difference')
axes[2].axhline(1.0, color='black', linestyle='--', label='No degradation')
axes[2].legend()

plt.suptitle('Noise Effect on Cross-Tissue Generalization\n(How much worse is performance on tissues different from training?)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Training tissue vs Test tissue at different noise levels
def create_cross_tissue_heatmap(train_tissue, metric='rmse'):
    """Create heatmap showing performance across tissues and noise levels."""
    data = cross_tissue_results[train_tissue]['results']
    
    # Create matrix: tissues x noise levels
    matrix = np.zeros((len(test_tissue_list), len(NOISE_SIGMAS)))
    
    for i, tissue in enumerate(test_tissue_list):
        matrix[i, :] = data[tissue][metric]
    
    return matrix


fig, axes = plt.subplots(1, len(training_tissues), figsize=(6*len(training_tissues), 6))

for idx, train_tissue in enumerate(training_tissues):
    matrix = create_cross_tissue_heatmap(train_tissue, metric='rmse')
    
    im = axes[idx].imshow(matrix, cmap='RdYlGn_r', aspect='auto')
    
    axes[idx].set_xticks(np.arange(len(NOISE_SIGMAS)))
    axes[idx].set_yticks(np.arange(len(test_tissue_list)))
    axes[idx].set_xticklabels([f'{s:.3f}' for s in NOISE_SIGMAS], rotation=45, ha='right')
    
    # Mark training tissue
    ylabels = []
    for tissue in test_tissue_list:
        if tissue == train_tissue:
            ylabels.append(f'→ {tissue} ←')
        else:
            ylabels.append(tissue)
    axes[idx].set_yticklabels(ylabels)
    
    axes[idx].set_xlabel('Noise Sigma')
    axes[idx].set_ylabel('Test Tissue')
    axes[idx].set_title(f'Trained on: {train_tissue}')
    
    plt.colorbar(im, ax=axes[idx], label='RMSE')

plt.suptitle('Cross-Tissue RMSE Heatmap (Green=Good, Red=Bad)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part C: Method Comparison Across Tissues

In [ ]:
def run_method_analysis(tissue_params, methods=GASP_METHODS,
                        noise_sigmas=NOISE_SIGMAS, n_mc=N_MONTE_CARLO):
    """Run noise analysis for all methods on a single tissue."""
    params = create_params(DEFAULT_NPCS)
    M_clean = generate_signal(T1=tissue_params['T1'], T2=tissue_params['T2'], params=params)
    
    results = {method: {'rmse': [], 'correlation': []} for method in methods}
    
    for sigma in noise_sigmas:
        for method in methods:
            rmse_list, corr_list = [], []
            
            for _ in range(n_mc):
                M_noisy = add_complex_noise(M_clean, sigma)
                
                try:
                    I, _ = train_gasp(M_noisy, D, method=method, useL2=True, lam=1e-2)
                    metrics = compute_metrics(I, D)
                    rmse_list.append(metrics['rmse'])
                    corr_list.append(metrics['correlation'])
                except:
                    rmse_list.append(np.nan)
                    corr_list.append(np.nan)
            
            results[method]['rmse'].append(np.nanmean(rmse_list))
            results[method]['correlation'].append(np.nanmean(corr_list))
    
    return results


# Run for all tissues
method_results = {}

print("Running method comparison across tissues...")
for tissue_name, tissue_params in tqdm(TISSUES.items(), desc="Tissues"):
    method_results[tissue_name] = run_method_analysis(tissue_params)

print("Done!")

In [ ]:
# Plot method comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

tissue_colors = cm.Set2(np.linspace(0, 1, len(TISSUES)))

for idx, method in enumerate(GASP_METHODS):
    for i, (tissue_name, results) in enumerate(method_results.items()):
        axes[idx].semilogy(NOISE_SIGMAS, results[method]['rmse'], 'o-',
                          color=tissue_colors[i], label=tissue_name, linewidth=2)
    
    axes[idx].set_xlabel('Noise Sigma')
    axes[idx].set_ylabel('RMSE (log)')
    axes[idx].set_title(f'{method.upper()} Method')
    axes[idx].legend(fontsize=9)

plt.suptitle('RMSE vs Noise by Method (Bandpass Target)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Aggregate comparison across methods
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

method_colors = cm.tab10(np.linspace(0, 1, len(GASP_METHODS)))

for i, method in enumerate(GASP_METHODS):
    # Average across all tissues
    rmse_all = np.array([method_results[t][method]['rmse'] for t in TISSUES.keys()])
    corr_all = np.array([method_results[t][method]['correlation'] for t in TISSUES.keys()])
    
    rmse_mean = np.mean(rmse_all, axis=0)
    rmse_std = np.std(rmse_all, axis=0)
    corr_mean = np.mean(corr_all, axis=0)
    corr_std = np.std(corr_all, axis=0)
    
    axes[0].semilogy(NOISE_SIGMAS, rmse_mean, 'o-', color=method_colors[i], 
                    label=method.upper(), linewidth=2)
    axes[0].fill_between(NOISE_SIGMAS, np.maximum(rmse_mean-rmse_std, 1e-4), 
                        rmse_mean+rmse_std, color=method_colors[i], alpha=0.2)
    
    axes[1].plot(NOISE_SIGMAS, corr_mean, 'o-', color=method_colors[i], 
                label=method.upper(), linewidth=2)
    axes[1].fill_between(NOISE_SIGMAS, corr_mean-corr_std, corr_mean+corr_std, 
                        color=method_colors[i], alpha=0.2)

axes[0].set_xlabel('Noise Sigma')
axes[0].set_ylabel('RMSE (log)')
axes[0].set_title('Average RMSE Across Tissues')
axes[0].legend()

axes[1].set_xlabel('Noise Sigma')
axes[1].set_ylabel('Correlation')
axes[1].set_title('Average Correlation Across Tissues')
axes[1].legend()
axes[1].set_ylim([0, 1.05])

plt.suptitle('Aggregate Method Comparison (Mean ± Std Across Tissues)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Summary

In [ ]:
print("="*80)
print("NOISE ROBUSTNESS ANALYSIS SUMMARY")
print("="*80)

print("\n1. PHASE CYCLE COUNT EFFECT:")
print("   - More phase cycles generally improve noise robustness")
print("   - Diminishing returns above 8 PCs for most tissues")
print(f"   - Tested: {PC_COUNTS} PCs with {len(TRS_BASE)} TRs")

print("\n2. CROSS-TISSUE GENERALIZATION:")
print("   - Performance degrades when T1/T2 ratio differs from training")
print("   - Noise amplifies cross-tissue degradation")
print("   - Training on intermediate ratios (like Gray Matter) provides balanced generalization")

# Find best/worst combinations
high_noise_df = degradation_df[degradation_df['is_same'] == False]
if len(high_noise_df) > 0:
    best = high_noise_df.loc[high_noise_df['corr_noisy'].idxmax()]
    worst = high_noise_df.loc[high_noise_df['corr_noisy'].idxmin()]
    print(f"\n   Best cross-tissue: Train={best['train_tissue']}, Test={best['test_tissue']}, Corr={best['corr_noisy']:.3f}")
    print(f"   Worst cross-tissue: Train={worst['train_tissue']}, Test={worst['test_tissue']}, Corr={worst['corr_noisy']:.3f}")

print("\n3. METHOD COMPARISON:")
# Find best method at high noise
noise_idx = 5
method_avg_corr = {}
for method in GASP_METHODS:
    corr_vals = [method_results[t][method]['correlation'][noise_idx] for t in TISSUES.keys()]
    method_avg_corr[method] = np.mean(corr_vals)

best_method = max(method_avg_corr, key=method_avg_corr.get)
print(f"   Best method at high noise (σ=0.05): {best_method.upper()}")
print(f"   Average correlation: {method_avg_corr[best_method]:.4f}")

print("\n" + "="*80)

In [ ]:
# Detailed summary table
print("\nDETAILED RESULTS TABLE")
print("="*90)
print(f"{'Tissue':<15} {'T1/T2':<8} {'Method':<12} {'RMSE(clean)':<12} {'RMSE(0.05)':<12} {'Corr(0.05)':<12}")
print("-"*90)

for tissue_name, results in method_results.items():
    ratio = TISSUES[tissue_name]['ratio']
    for method in GASP_METHODS:
        rmse_clean = results[method]['rmse'][0]
        rmse_noisy = results[method]['rmse'][5]
        corr_noisy = results[method]['correlation'][5]
        print(f"{tissue_name:<15} {ratio:<8.1f} {method:<12} {rmse_clean:<12.4f} {rmse_noisy:<12.4f} {corr_noisy:<12.4f}")

In [ ]:
print("\nAnalysis complete!")